# 2 · Pragmatic Classification — Model Comparison

**Measuring Pragmatic Alignment in LLM-Based Agents**
University of Trier · NLP Master's Program · WS 2025/26

Trains and compares six approaches on the four pragmatic dimensions
(STANCE · ACTION · PERSONALNESS · POLITENESS), using one stratified split
shared by every model so the numbers are directly comparable.

| Step | What it does |
|------|-------------|
| 1 | Load the prepared dataset and encode labels |
| 2 | Build one stratified train/val/test split, reused by all models |
| 3 | Encode text with SBERT (frozen sentence embeddings) |
| 4 | Majority baseline · TF-IDF + LinearSVC · SBERT + LinearSVC · SBERT + Logistic Regression · SBERT + XGBoost |
| 5 | Fine-tune MiniLM-L6 and RoBERTa-base end-to-end on the same split |
| 6 | Comparative results table |
| 7 | Serialise the XGBoost critics + label encoders for the agentic pipeline |

In [1]:
import os

# On macOS, xgboost and torch each bundle an OpenMP runtime; loading both
# unguarded crashes the interpreter. Pin the thread count before either is
# imported, and import xgboost first so its runtime is the one in use.
os.environ.setdefault("OMP_NUM_THREADS", "1")

from xgboost import XGBClassifier  # noqa: E402  (must precede torch)

import json  # noqa: E402
import random  # noqa: E402
import warnings  # noqa: E402
from pathlib import Path  # noqa: E402

import joblib  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA = Path("../data/annotated_clean.csv")
OUT = Path("../notebooks")
LABEL_COLS = ["STANCE", "ACTION", "PERSONALNESS", "POLITENESS"]

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"seed={SEED}  device={DEVICE}")

seed=42  device=mps


---
## 1. Load data and encode labels

The classifier input is the target tweet concatenated with the reply, so the
model sees the reply *in context* — a reply's pragmatic function depends on
what it is responding to.

In [2]:
df = pd.read_csv(DATA)
df["text"] = df["target_tweet"].astype(str) + " " + df["authentic_reply"].astype(str)
print(f"rows: {len(df)}")

encoders, Y = {}, pd.DataFrame(index=df.index)
for col in LABEL_COLS:
    le = LabelEncoder().fit(df[col])
    encoders[col] = le
    Y[col] = le.transform(df[col])
    print(f"{col:13} {len(le.classes_)} classes: {list(le.classes_)}")

rows: 800
STANCE        3 classes: ['NEUTRAL', 'OPPOSE', 'SUPPORT']
ACTION        4 classes: ['COMMAND', 'QUESTION', 'REACTION', 'STATEMENT']
PERSONALNESS  2 classes: ['GENERAL', 'PERSONAL']
POLITENESS    3 classes: ['NORMAL', 'POLITE', 'RUDE']


---
## 2. One stratified split, shared by every model

The dataset is small and the label distributions are skewed, so an unstratified
split can leave a rare class absent from the test set entirely. We stratify on
the joint label combination, collapsing combinations too rare to split into a
single `RARE` stratum.

Every model below is trained and evaluated on **this same split**, so the
comparison is like-for-like.

In [3]:
joint = df[LABEL_COLS].agg("|".join, axis=1)
counts = joint.value_counts()
strata = joint.where(joint.map(counts) >= 8, "RARE")
print(f"{strata.nunique()} strata (from {joint.nunique()} raw label combinations)")

idx_train, idx_temp = train_test_split(
    df.index, test_size=0.30, random_state=SEED, stratify=strata
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=SEED, stratify=strata[idx_temp]
)
print(f"train {len(idx_train)} · val {len(idx_val)} · test {len(idx_test)}")

print("\nClass distribution per split (test counts in brackets):")
for col in LABEL_COLS:
    parts = [
        f"{cls} {(df.loc[idx_train, col] == cls).sum()}/{(df.loc[idx_val, col] == cls).sum()}/{(df.loc[idx_test, col] == cls).sum()}"
        for cls in encoders[col].classes_
    ]
    print(f"  {col:13} " + "  ".join(parts))

X_train_txt = df.loc[idx_train, "text"].tolist()
X_val_txt = df.loc[idx_val, "text"].tolist()
X_test_txt = df.loc[idx_test, "text"].tolist()
Y_train, Y_val, Y_test = Y.loc[idx_train], Y.loc[idx_val], Y.loc[idx_test]

19 strata (from 38 raw label combinations)
train 560 · val 120 · test 120

Class distribution per split (test counts in brackets):
  STANCE        NEUTRAL 172/36/38  OPPOSE 253/54/53  SUPPORT 135/30/29
  ACTION        COMMAND 60/16/13  QUESTION 83/19/17  REACTION 14/3/4  STATEMENT 403/82/86
  PERSONALNESS  GENERAL 468/102/104  PERSONAL 92/18/16
  POLITENESS    NORMAL 434/95/90  POLITE 16/3/6  RUDE 110/22/24


---
## 3. SBERT sentence embeddings

`all-MiniLM-L6-v2` produces a frozen 384-dimensional sentence embedding. These
features are shared by the feature-based classifiers below.

In [4]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
X_train_emb = embedder.encode(X_train_txt, show_progress_bar=False, batch_size=64)
X_val_emb = embedder.encode(X_val_txt, show_progress_bar=False, batch_size=64)
X_test_emb = embedder.encode(X_test_txt, show_progress_bar=False, batch_size=64)
print("embeddings:", X_train_emb.shape, X_val_emb.shape, X_test_emb.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embeddings: (560, 384) (120, 384) (120, 384)


---
## 4. Evaluation helper

Accuracy alone is misleading on skewed labels — a model that always predicts
the majority class scores well. We therefore report **macro-F1** (unweighted
mean over classes) alongside accuracy for every dimension.

In [5]:
RESULTS = {}

def evaluate(name, preds):
    """Score per-dimension predictions and store the summary."""
    rows = []
    for col in LABEL_COLS:
        y_true, y_pred = Y_test[col].values, preds[col]
        rows.append({
            "Dimension": col,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "Weighted-F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        })
    table = pd.DataFrame(rows).set_index("Dimension")
    table.loc["MEAN"] = table.mean()
    RESULTS[name] = table
    print(f"\n=== {name} ===")
    print(table.round(4).to_string())
    return table

---
## 5. Reference point: majority-class baseline

Three of the four dimensions are heavily skewed (STATEMENT 71%, GENERAL 84%,
NORMAL 77%). A classifier that ignores the input entirely and always predicts
the most frequent training class therefore already scores highly on accuracy.

This baseline is the number every model below has to beat for its accuracy to
mean anything.

In [6]:
from sklearn.dummy import DummyClassifier

preds = {}
for col in LABEL_COLS:
    clf = DummyClassifier(strategy="most_frequent")
    clf.fit(X_train_emb, Y_train[col])
    preds[col] = clf.predict(X_test_emb)
evaluate("Majority-class baseline", preds)


=== Majority-class baseline ===
              Accuracy  Macro-F1  Weighted-F1
Dimension                                    
STANCE          0.4417    0.2042       0.2706
ACTION          0.7167    0.2087       0.5984
PERSONALNESS    0.8667    0.4643       0.8048
POLITENESS      0.7500    0.2857       0.6429
MEAN            0.6938    0.2907       0.5792


,Accuracy,Macro-F1,Weighted-F1
Dimension,,,
STANCE,0.441667,0.204239,0.270617
ACTION,0.716667,0.208738,0.598382
PERSONALNESS,0.866667,0.464286,0.804762
POLITENESS,0.750000,0.285714,0.642857
MEAN,0.693750,0.290744,0.579154


---
## 6. Lexical baseline: TF-IDF + Linear SVM

A purely lexical baseline: can pragmatic function be read off surface word
choice alone?

In [7]:
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train_txt)
X_test_tfidf = tfidf.transform(X_test_txt)

preds = {}
for col in LABEL_COLS:
    clf = LinearSVC(class_weight="balanced", random_state=SEED)
    clf.fit(X_train_tfidf, Y_train[col])
    preds[col] = clf.predict(X_test_tfidf)
evaluate("TF-IDF + LinearSVC", preds)


=== TF-IDF + LinearSVC ===
              Accuracy  Macro-F1  Weighted-F1
Dimension                                    
STANCE          0.4333    0.3869       0.4144
ACTION          0.6833    0.2441       0.6077
PERSONALNESS    0.8500    0.5496       0.8193
POLITENESS      0.7333    0.3425       0.6707
MEAN            0.6750    0.3808       0.6280


,Accuracy,Macro-F1,Weighted-F1
Dimension,,,
STANCE,0.433333,0.386853,0.414353
ACTION,0.683333,0.244082,0.607701
PERSONALNESS,0.850000,0.549625,0.819349
POLITENESS,0.733333,0.342530,0.670692
MEAN,0.675000,0.380772,0.628024


---
## 7. SBERT + Linear SVM

In [8]:
preds = {}
for col in LABEL_COLS:
    clf = LinearSVC(class_weight="balanced", random_state=SEED)
    clf.fit(X_train_emb, Y_train[col])
    preds[col] = clf.predict(X_test_emb)
evaluate("SBERT + LinearSVC", preds)


=== SBERT + LinearSVC ===
              Accuracy  Macro-F1  Weighted-F1
Dimension                                    
STANCE          0.4167    0.3663       0.3986
ACTION          0.6250    0.3099       0.6190
PERSONALNESS    0.7333    0.5200       0.7547
POLITENESS      0.7333    0.3936       0.6978
MEAN            0.6271    0.3974       0.6175


,Accuracy,Macro-F1,Weighted-F1
Dimension,,,
STANCE,0.416667,0.366275,0.398640
ACTION,0.625000,0.309868,0.619032
PERSONALNESS,0.733333,0.520000,0.754667
POLITENESS,0.733333,0.393614,0.697826
MEAN,0.627083,0.397439,0.617541


---
## 8. SBERT + Logistic Regression

In [9]:
preds = {}
for col in LABEL_COLS:
    clf = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)
    clf.fit(X_train_emb, Y_train[col])
    preds[col] = clf.predict(X_test_emb)
evaluate("SBERT + LogisticRegression", preds)


=== SBERT + LogisticRegression ===
              Accuracy  Macro-F1  Weighted-F1
Dimension                                    
STANCE          0.4583    0.4120       0.4525
ACTION          0.5167    0.3509       0.5575
PERSONALNESS    0.7000    0.4991       0.7317
POLITENESS      0.6667    0.4653       0.6859
MEAN            0.5854    0.4318       0.6069


,Accuracy,Macro-F1,Weighted-F1
Dimension,,,
STANCE,0.458333,0.411988,0.452517
ACTION,0.516667,0.350926,0.557469
PERSONALNESS,0.700000,0.499072,0.731725
POLITENESS,0.666667,0.465292,0.685890
MEAN,0.585417,0.431820,0.606900


---
## 9. SBERT + XGBoost

One gradient-boosted classifier per dimension. These are the models reused as
**critics** in the agentic pipeline (notebook 4), so they are trained with a
probabilistic objective (`multi:softprob` / `binary:logistic`) — the pipeline
needs calibrated class probabilities, not just hard labels.

In [10]:
xgb_models, preds = {}, {}
for col in LABEL_COLS:
    clf = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=SEED,
    )
    clf.fit(X_train_emb, Y_train[col])
    xgb_models[col] = clf
    preds[col] = clf.predict(X_test_emb)
xgb_table = evaluate("SBERT + XGBoost", preds)

print("\nPer-class detail (XGBoost):")
for col in LABEL_COLS:
    print(f"\n--- {col} ---")
    print(classification_report(
        Y_test[col], preds[col],
        labels=list(range(len(encoders[col].classes_))),
        target_names=encoders[col].classes_, zero_division=0,
    ))


=== SBERT + XGBoost ===
              Accuracy  Macro-F1  Weighted-F1
Dimension                                    
STANCE          0.4583    0.3765       0.4198
ACTION          0.7083    0.2094       0.6002
PERSONALNESS    0.8667    0.5195       0.8190
POLITENESS      0.7500    0.3510       0.6797
MEAN            0.6958    0.3641       0.6297

Per-class detail (XGBoost):

--- STANCE ---
              precision    recall  f1-score   support

     NEUTRAL       0.25      0.16      0.19        38
      OPPOSE       0.57      0.79      0.66        53
     SUPPORT       0.32      0.24      0.27        29

    accuracy                           0.46       120
   macro avg       0.38      0.40      0.38       120
weighted avg       0.41      0.46      0.42       120


--- ACTION ---
              precision    recall  f1-score   support

     COMMAND       0.00      0.00      0.00        13
    QUESTION       0.00      0.00      0.00        17
    REACTION       0.00      0.00      0.00     

---
## 10. End-to-end transformer fine-tuning

A shared encoder with four classification heads, fine-tuned jointly on all four
dimensions and evaluated on the same split as the models above.

Two encoders are fine-tuned:

- **MiniLM-L6** — the *same* encoder used for the frozen embeddings in sections
  7–9. Comparing it against `SBERT + XGBoost` isolates the effect of the
  training regime, since the encoder is held constant.
- **RoBERTa-base** — a substantially larger encoder (125M parameters), included
  to check that the finding is not an artefact of model capacity.

In [11]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

MAX_LEN, BATCH, EPOCHS, LR = 128, 16, 5, 2e-5
NUM_CLASSES = [len(encoders[c].classes_) for c in LABEL_COLS]

class PragmaticDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts, self.tokenizer = list(texts), tokenizer
        self.labels = np.asarray(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i], truncation=True, padding="max_length",
            max_length=MAX_LEN, return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

class MultiHeadTransformer(torch.nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = torch.nn.Dropout(0.1)
        self.heads = torch.nn.ModuleList([torch.nn.Linear(hidden, n) for n in num_classes])

    def forward(self, input_ids, attention_mask, **kwargs):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return [head(self.dropout(pooled)) for head in self.heads]

def finetune(label, model_name):
    """Fine-tune `model_name` end-to-end and evaluate it on the shared test set."""
    torch.manual_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = MultiHeadTransformer(model_name, NUM_CLASSES).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"\n--- {label} ({model_name}, {n_params:.0f}M params) ---")

    train_dl = DataLoader(
        PragmaticDataset(X_train_txt, Y_train.values, tokenizer), batch_size=BATCH, shuffle=True
    )
    test_dl = DataLoader(
        PragmaticDataset(X_test_txt, Y_test.values, tokenizer), batch_size=BATCH
    )

    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    loss_fn = torch.nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):
        model.train()
        total = 0.0
        for batch in train_dl:
            labels = batch.pop("labels").to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optim.zero_grad()
            logits = model(**batch)
            loss = sum(loss_fn(logits[i], labels[:, i]) for i in range(len(LABEL_COLS)))
            loss.backward()
            optim.step()
            total += loss.item()
        print(f"  epoch {epoch + 1}/{EPOCHS}  train loss {total / len(train_dl):.4f}")

    model.eval()
    collected = [[] for _ in LABEL_COLS]
    with torch.no_grad():
        for batch in test_dl:
            batch.pop("labels")
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            for i, logit in enumerate(model(**batch)):
                collected[i].append(logit.argmax(-1).cpu().numpy())

    evaluate(label, {c: np.concatenate(collected[i]) for i, c in enumerate(LABEL_COLS)})
    del model
    torch.mps.empty_cache() if DEVICE == "mps" else None

for label, model_name in [
    ("MiniLM-L6 (fine-tuned)", "sentence-transformers/all-MiniLM-L6-v2"),
    ("RoBERTa-base (fine-tuned)", "roberta-base"),
]:
    try:
        finetune(label, model_name)
    except Exception as exc:  # model unavailable offline -> skip rather than abort
        print(f"\n--- {label}: skipped ({type(exc).__name__}: {str(exc)[:100]}) ---")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- MiniLM-L6 (fine-tuned) (sentence-transformers/all-MiniLM-L6-v2, 23M params) ---


  epoch 1/5  train loss 3.3809


  epoch 2/5  train loss 2.9712


  epoch 3/5  train loss 2.8511


...

  epoch 2/5  train loss 2.9007


  epoch 3/5  train loss 2.4634


  epoch 4/5  train loss 1.9290


  epoch 5/5  train loss 1.4546



=== RoBERTa-base (fine-tuned) ===
              Accuracy  Macro-F1  Weighted-F1
Dimension                                    
STANCE          0.5500    0.5280       0.5482
ACTION          0.8000    0.4513       0.7590
PERSONALNESS    0.8250    0.6311       0.8272
POLITENESS      0.7917    0.4660       0.7591
MEAN            0.7417    0.5191       0.7234


> **Reproducibility note.** The feature-based models (sections 5–9) and the
> MiniLM fine-tune are exactly reproducible: re-running this notebook reproduces
> their numbers to four decimal places. End-to-end fine-tuning of RoBERTa-base is
> *not* bit-reproducible on Apple-Silicon (MPS), because the backward pass over
> its embedding table uses non-deterministic atomic accumulation. Two independent
> runs of this notebook gave mean accuracy 0.742 and 0.765 (macro-F1 0.519 and
> 0.575). The ordering of the models is stable across runs; the exact RoBERTa
> figures should be read as approximate to roughly ±0.03.
>
> All figures quoted in the report are the ones stored in this notebook's outputs.

---
## 11. Comparative results

In [12]:
summary = pd.DataFrame({
    name: {"Accuracy": t.loc["MEAN", "Accuracy"], "Macro-F1": t.loc["MEAN", "Macro-F1"]}
    for name, t in RESULTS.items()
}).T.sort_values("Macro-F1", ascending=False)
print("Mean across the four pragmatic dimensions:\n")
print(summary.round(4).to_string())

print("\n\nPer-dimension accuracy:\n")
print(pd.DataFrame({n: t["Accuracy"].drop("MEAN") for n, t in RESULTS.items()}).round(4).to_string())
print("\n\nPer-dimension macro-F1:\n")
print(pd.DataFrame({n: t["Macro-F1"].drop("MEAN") for n, t in RESULTS.items()}).round(4).to_string())

Mean across the four pragmatic dimensions:

                            Accuracy  Macro-F1
RoBERTa-base (fine-tuned)     0.7417    0.5191
SBERT + LogisticRegression    0.5854    0.4318
SBERT + LinearSVC             0.6271    0.3974
MiniLM-L6 (fine-tuned)        0.7292    0.3823
TF-IDF + LinearSVC            0.6750    0.3808
SBERT + XGBoost               0.6958    0.3641
Majority-class baseline       0.6938    0.2907


Per-dimension accuracy:

              Majority-class baseline  TF-IDF + LinearSVC  SBERT + LinearSVC  SBERT + LogisticRegression  SBERT + XGBoost  MiniLM-L6 (fine-tuned)  RoBERTa-base (fine-tuned)
Dimension                                                                                                                                                                   
STANCE                         0.4417              0.4333             0.4167                      0.4583           0.4583                  0.5583                     0.5500
ACTION                         0.7

---
## 12. Serialise critics and label encoders

The four XGBoost models are saved as `xgb_output_0..3.json` in the dimension
order `STANCE · ACTION · PERSONALNESS · POLITENESS`, matching the order the
agentic pipeline (notebook 4) loads them in.

In [13]:
for i, col in enumerate(LABEL_COLS):
    path = OUT / f"xgb_output_{i}.json"
    xgb_models[col].save_model(path)
    print(f"{col:13} -> {path.name}  ({len(encoders[col].classes_)} classes)")

joblib.dump(encoders, OUT / "label_encoders.pkl")
print(f"encoders      -> label_encoders.pkl")

(OUT / "test_split_indices.json").write_text(json.dumps({
    "seed": SEED,
    "train": [int(i) for i in idx_train],
    "val": [int(i) for i in idx_val],
    "test": [int(i) for i in idx_test],
}))
print("split indices -> test_split_indices.json")

STANCE        -> xgb_output_0.json  (3 classes)
ACTION        -> xgb_output_1.json  (4 classes)
PERSONALNESS  -> xgb_output_2.json  (2 classes)
POLITENESS    -> xgb_output_3.json  (3 classes)
encoders      -> label_encoders.pkl
split indices -> test_split_indices.json
